# Chapter 2: Core Agentic Architectures and Mechanisms

## Production-Grade Design Patterns for Multi-Agent EDA Systems

---

> *"Production-grade agentic systems require explicit control over reasoning paths. Unlike early 'black box' autonomous agents, 2026-standard systems utilize Agentic Workflows modeled as stateful graphs."*

### Learning Objectives

1. **Implement** the four fundamental MAS design patterns: Supervisor-Worker, Consensus-Based, Handoff, and Stateful Graph
2. **Analyze** tradeoffs between patterns in terms of latency, fault tolerance, and design quality
3. **Build** a working Supervisor-Worker system with typed state management
4. **Design** consensus mechanisms for topology selection using PPA arbitration
5. **Implement** dynamic handoff protocols with error context propagation
6. **Model** design flows as Directed Acyclic Graphs (DAGs) and cyclic loops

---

## 2.1 Pattern Overview: The Four Pillars of MAS Architecture

Modern MAS for EDA are built on four fundamental orchestration patterns. Each pattern has distinct properties that make it suitable for specific phases of the design flow.

| Pattern | Control Flow | Best For | Latency | Fault Tolerance |
|---------|-------------|----------|---------|----------------|
| **Supervisor-Worker** | Centralized | Task decomposition, parallel execution | Medium | High (supervisor can reassign) |
| **Consensus-Based** | Distributed | Topology exploration, multi-objective optimization | High | Very High (no single point of failure) |
| **Handoff** | Sequential | Pipeline stages, error escalation | Low per step | Medium (chain dependency) |
| **Stateful Graph** | Graph-driven | Complex flows with conditionals and loops | Variable | High (checkpoint-based recovery) |

### When to Use Each Pattern

- **Supervisor-Worker**: When you need a central coordinator that decomposes a high-level goal (e.g., "Design a low-power RISC-V core") into independent sub-tasks
- **Consensus-Based**: When exploring design alternatives where multiple valid solutions exist and you need Pareto-optimal selection
- **Handoff**: When tasks are inherently sequential and context must flow from one specialist to the next
- **Stateful Graph**: When the flow has conditional branches, loops, and requires checkpointing for long-running simulations

In [ ]:
import sys; sys.path.insert(0, '..')
from style_utils import (setup_3b1b_style, glow_line, glow_fill, styled_box,
                         styled_arrow, finish_plot, plotly_3b1b_layout,
                         BACKGROUND, SURFACE, TEXT, TEXT_DIM, GRID,
                         BLUE, TEAL, GREEN, YELLOW, GOLD, RED,
                         ROSE, PURPLE, CYAN, ORANGE, PALETTE)
setup_3b1b_style()

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for ax in axes.flat:
    ax.axis('off')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)

# Pattern 1: Supervisor-Worker
ax1 = axes[0, 0]
ax1.set_title('Supervisor-Worker Pattern', fontsize=13,
              fontweight='bold', color=GREEN, pad=10)
styled_box(ax1, 3.5, 8, 3, 1.2, 'Supervisor', 'Orchestrator', GREEN)
styled_box(ax1, 0.5, 5, 2.5, 1.2, 'Synthesis', 'Agent', TEAL)
styled_box(ax1, 3.75, 5, 2.5, 1.2, 'DFT', 'Agent', BLUE)
styled_box(ax1, 7, 5, 2.5, 1.2, 'Timing', 'Agent', YELLOW)
styled_box(ax1, 3.5, 2, 3, 1.2, 'Result', 'Aggregation', RED)
for x_pos in [1.75, 5, 8.25]:
    styled_arrow(ax1, 5, 8, x_pos, 6.2, color=GREEN)
    styled_arrow(ax1, x_pos, 5, 5, 3.2, color=YELLOW)

# Pattern 2: Consensus-Based
ax2 = axes[0, 1]
ax2.set_title('Consensus-Based Reasoning', fontsize=13,
              fontweight='bold', color=YELLOW, pad=10)
styled_box(ax2, 3.5, 8, 3, 1.2, 'Design Goal', 'Multi-objective', YELLOW)
styled_box(ax2, 0.5, 5, 2.5, 1.2, 'Proposal A', 'Folded Cascode', TEAL)
styled_box(ax2, 3.75, 5, 2.5, 1.2, 'Proposal B', 'Telescopic', BLUE)
styled_box(ax2, 7, 5, 2.5, 1.2, 'Proposal C', 'Two-Stage', ROSE)
styled_box(ax2, 3.5, 2, 3, 1.2, 'PPA Arbiter', 'Selects optimal', YELLOW)
for x_pos in [1.75, 5, 8.25]:
    styled_arrow(ax2, 5, 8, x_pos, 6.2, color=YELLOW)
    styled_arrow(ax2, x_pos, 5, 5, 3.2, color=TEAL)

# Pattern 3: Handoff
ax3 = axes[1, 0]
ax3.set_title('Handoff Pattern', fontsize=13,
              fontweight='bold', color=RED, pad=10)
agent_defs = [
    (1, 5.5, 'Topology', 'Selection', TEAL),
    (3.5, 5.5, 'Sizing', 'Agent', BLUE),
    (6, 5.5, 'Simulation', 'Agent', YELLOW),
    (8.5, 5.5, 'Layout', 'Agent', ROSE),
]
for bx, by, blabel, bsub, bcolor in agent_defs:
    styled_box(ax3, bx - 0.5, by, 2, 1.3, blabel, bsub, bcolor)
for k in range(len(agent_defs) - 1):
    styled_arrow(ax3, agent_defs[k][0] + 1.5, 6.15,
                 agent_defs[k + 1][0] - 0.5, 6.15, color=TEXT)
styled_arrow(ax3, 6.5, 5.5, 3.5, 5.5, color=RED)
ax3.text(5, 4.8, 'FAIL: handoff back with error context', fontsize=8,
         color=RED, ha='center', style='italic')

# Pattern 4: Stateful Graph
ax4 = axes[1, 1]
ax4.set_title('Stateful Graph Workflow (DAG)', fontsize=13,
              fontweight='bold', color=PURPLE, pad=10)
graph_nodes = [
    (5, 9, 'START', '', TEXT_DIM),
    (5, 7.5, 'Supervisor', '', GREEN),
    (2.5, 5.5, 'Synthesis', '', TEAL),
    (7.5, 5.5, 'Verification', '', YELLOW),
    (5, 3.5, 'Merge', '', BLUE),
    (5, 1.5, 'Layout', '', ROSE),
]
for gx, gy, glabel, gsub, gcolor in graph_nodes:
    styled_box(ax4, gx - 1, gy, 2, 0.9, glabel, gsub, gcolor)
styled_arrow(ax4, 5, 9, 5, 8.4, color=TEXT)
styled_arrow(ax4, 5, 7.5, 2.5, 6.4, color=GREEN)
styled_arrow(ax4, 5, 7.5, 7.5, 6.4, color=GREEN)
styled_arrow(ax4, 2.5, 5.5, 5, 4.4, color=TEAL)
styled_arrow(ax4, 7.5, 5.5, 5, 4.4, color=YELLOW)
styled_arrow(ax4, 5, 3.5, 5, 2.4, color=TEXT)
styled_arrow(ax4, 4, 3.5, 4, 7.5, color=RED)
ax4.text(2.8, 5, 'FAIL ->\nloop back', fontsize=7, color=RED,
         ha='center', style='italic')

fig.suptitle('The Four Pillars of Multi-Agent Architecture for EDA',
             fontsize=16, fontweight='bold', color=TEXT, y=1.02)
finish_plot(fig)
plt.show()

## 2.2 The Supervisor-Worker Pattern — Deep Implementation

### Architecture

The Supervisor-Worker pattern uses a **central orchestrator** (the Supervisor) powered by a frontier LLM (e.g., GPT-5.2 or Claude 4.5) that:

1. **Receives** a high-level design intent
2. **Decomposes** it into typed sub-tasks
3. **Dispatches** sub-tasks to specialized worker agents
4. **Aggregates** results and determines next actions
5. **Iterates** if any sub-task fails convergence criteria

### Formal Specification

$$\text{Supervisor}: \mathcal{I} \rightarrow \{(t_1, a_1), (t_2, a_2), \ldots, (t_k, a_k)\}$$

Where $\mathcal{I}$ is the intent space, $t_i$ is a typed task, and $a_i \in \mathcal{A}$ is the assigned agent.

The aggregation function:

$$\text{Aggregate}: \{r_1, r_2, \ldots, r_k\} \rightarrow (\mathcal{S}', d)$$

Maps worker results to an updated state $\mathcal{S}'$ and a decision $d \in \{\text{proceed}, \text{iterate}, \text{escalate}\}$.

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Optional
from enum import Enum
import json
import uuid
from datetime import datetime


class TaskStatus(Enum):
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"


class AgentRole(Enum):
    SUPERVISOR = "supervisor"
    TOPOLOGY = "topology_selection"
    SIZING = "transistor_sizing"
    SIMULATION = "spice_simulation"
    VERIFICATION = "design_verification"
    LAYOUT = "physical_layout"
    CRITIC = "design_critic"


@dataclass
class DesignConstraints:
    """Typed representation of analog design constraints."""
    supply_voltage: float  # VDD in volts
    dc_gain_min: float  # Minimum DC gain in dB
    gbw_min: float  # Minimum gain-bandwidth product in MHz
    phase_margin_min: float  # Minimum phase margin in degrees
    power_max: float  # Maximum power in mW
    load_capacitance: float  # Load capacitance in pF
    technology_node: str  # e.g., "sky130"
    temperature_range: tuple[float, float] = (-40, 125)  # °C


@dataclass
class AgentTask:
    """A typed task dispatched by the Supervisor."""
    task_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    assigned_agent: AgentRole = AgentRole.SUPERVISOR
    description: str = ""
    constraints: Optional[DesignConstraints] = None
    input_data: dict = field(default_factory=dict)
    status: TaskStatus = TaskStatus.PENDING
    result: Optional[dict] = None
    error_context: Optional[str] = None
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    iteration: int = 0


@dataclass
class DesignState:
    """Shared mutable state for the MAS."""
    intent: str = ""
    constraints: Optional[DesignConstraints] = None
    topology: Optional[str] = None
    netlist: Optional[str] = None
    simulation_results: Optional[dict] = None
    ppa_metrics: Optional[dict] = None
    layout: Optional[dict] = None
    tasks: list[AgentTask] = field(default_factory=list)
    iteration_count: int = 0
    converged: bool = False
    history: list[dict] = field(default_factory=list)


print("=" * 60)
print("  Typed State System for Multi-Agent EDA")
print("=" * 60)

constraints = DesignConstraints(
    supply_voltage=1.8,
    dc_gain_min=60,
    gbw_min=100,
    phase_margin_min=60,
    power_max=1.0,
    load_capacitance=5.0,
    technology_node="sky130"
)

state = DesignState(
    intent="Design a two-stage Miller-compensated OTA with 60dB gain and 100MHz GBW",
    constraints=constraints
)

print(f"\nDesign Intent: {state.intent}")
print(f"\nConstraints:")
print(f"  VDD:            {constraints.supply_voltage}V")
print(f"  DC Gain:        ≥{constraints.dc_gain_min} dB")
print(f"  GBW:            ≥{constraints.gbw_min} MHz")
print(f"  Phase Margin:   ≥{constraints.phase_margin_min}°")
print(f"  Power:          ≤{constraints.power_max} mW")
print(f"  C_load:         {constraints.load_capacitance} pF")
print(f"  Technology:     {constraints.technology_node}")
print(f"  Temp Range:     {constraints.temperature_range}")

In [ ]:
from abc import ABC, abstractmethod
import time
import random


class BaseAgent(ABC):
    """Abstract base agent with common lifecycle methods."""
    
    def __init__(self, role: AgentRole, model: str = "claude-4.5-sonnet"):
        self.role = role
        self.model = model
        self.call_count = 0
    
    @abstractmethod
    def execute(self, task: AgentTask, state: DesignState) -> dict:
        """Execute the agent's core function."""
        pass
    
    def log(self, message: str):
        print(f"  [{self.role.value}] {message}")


class TopologyAgent(BaseAgent):
    """Selects optimal circuit topology based on constraints."""
    
    TOPOLOGIES = {
        "telescopic_cascode": {
            "max_gain_db": 80, "max_gbw_mhz": 500, "power_efficiency": 0.9,
            "output_swing": "low", "complexity": "medium"
        },
        "folded_cascode": {
            "max_gain_db": 70, "max_gbw_mhz": 300, "power_efficiency": 0.7,
            "output_swing": "medium", "complexity": "medium"
        },
        "two_stage_miller": {
            "max_gain_db": 100, "max_gbw_mhz": 200, "power_efficiency": 0.5,
            "output_swing": "high", "complexity": "high"
        },
        "gain_boosted_cascode": {
            "max_gain_db": 120, "max_gbw_mhz": 150, "power_efficiency": 0.4,
            "output_swing": "medium", "complexity": "very_high"
        }
    }
    
    def __init__(self):
        super().__init__(AgentRole.TOPOLOGY)
    
    def execute(self, task: AgentTask, state: DesignState) -> dict:
        self.call_count += 1
        self.log("Analyzing design constraints for topology selection...")
        
        c = state.constraints
        candidates = []
        
        for name, specs in self.TOPOLOGIES.items():
            if specs["max_gain_db"] >= c.dc_gain_min and specs["max_gbw_mhz"] >= c.gbw_min:
                score = (
                    specs["power_efficiency"] * 0.3 +
                    (specs["max_gain_db"] / 120) * 0.3 +
                    (specs["max_gbw_mhz"] / 500) * 0.2 +
                    (1 if specs["output_swing"] == "high" else 0.5) * 0.2
                )
                candidates.append((name, score, specs))
                self.log(f"  Candidate: {name} (score={score:.3f})")
        
        if not candidates:
            return {"status": "failed", "error": "No topology meets all constraints"}
        
        candidates.sort(key=lambda x: x[1], reverse=True)
        selected = candidates[0]
        
        self.log(f"  Selected: {selected[0]} (score={selected[1]:.3f})")
        
        return {
            "status": "success",
            "topology": selected[0],
            "score": selected[1],
            "specs": selected[2],
            "alternatives": [(c[0], c[1]) for c in candidates[1:]]
        }


class SizingAgent(BaseAgent):
    """Determines transistor W/L ratios and bias currents."""
    
    def __init__(self):
        super().__init__(AgentRole.SIZING)
    
    def execute(self, task: AgentTask, state: DesignState) -> dict:
        self.call_count += 1
        self.log("Computing transistor sizes for selected topology...")
        
        c = state.constraints
        topology = state.topology
        
        # Simplified gm/Id methodology for two-stage Miller OTA
        gm_id_target = 15  # V^-1 (moderate inversion)
        CL = c.load_capacitance * 1e-12
        GBW = c.gbw_min * 1e6 * 2 * 3.14159
        
        gm1 = GBW * CL  # First stage transconductance
        Id1 = gm1 / gm_id_target
        
        # W/L from gm/Id lookup (simplified)
        mu_n_cox = 270e-6  # SKY130 NMOS
        L = 0.5e-6  # 500nm for gain
        W1 = (2 * Id1) / (mu_n_cox * (gm1 / Id1)**(-2) * L)  # Simplified
        W1 = max(W1, 2e-6)  # Minimum 2um
        
        # Compensation capacitor (Miller)
        Cc = 0.25 * CL  # Typical: Cc = CL/4
        
        sizing = {
            "M1_M2": {"W": f"{W1*1e6:.1f}u", "L": f"{L*1e6:.1f}u", "type": "NMOS", "role": "diff_pair"},
            "M3_M4": {"W": f"{W1*2*1e6:.1f}u", "L": f"{L*1e6:.1f}u", "type": "PMOS", "role": "active_load"},
            "M5": {"W": f"{W1*1e6:.1f}u", "L": f"{L*1e6:.1f}u", "type": "NMOS", "role": "tail_current"},
            "M6": {"W": f"{W1*4*1e6:.1f}u", "L": "0.3u", "type": "PMOS", "role": "second_stage"},
            "M7": {"W": f"{W1*2*1e6:.1f}u", "L": "0.3u", "type": "NMOS", "role": "second_stage_load"},
            "Cc": f"{Cc*1e12:.2f}pF",
            "bias_current": f"{Id1*1e6:.1f}uA",
            "gm1": f"{gm1*1e3:.2f}mS"
        }
        
        self.log(f"  gm1 = {gm1*1e3:.2f} mS")
        self.log(f"  Id1 = {Id1*1e6:.1f} uA")
        self.log(f"  Cc  = {Cc*1e12:.2f} pF")
        self.log(f"  Sizing complete for {topology}")
        
        return {"status": "success", "sizing": sizing}


class SimulationAgent(BaseAgent):
    """Runs SPICE simulation and extracts performance metrics."""
    
    def __init__(self):
        super().__init__(AgentRole.SIMULATION)
    
    def execute(self, task: AgentTask, state: DesignState) -> dict:
        self.call_count += 1
        self.log("Running SPICE simulation (AC, DC, Transient)...")
        
        # Simulated results (in production, this calls ngspice via MCP)
        c = state.constraints
        noise_factor = random.uniform(0.85, 1.15)
        
        results = {
            "dc_gain_db": c.dc_gain_min * noise_factor * 1.1,
            "gbw_mhz": c.gbw_min * noise_factor,
            "phase_margin_deg": c.phase_margin_min * noise_factor * 1.05,
            "power_mw": c.power_max * noise_factor * 0.7,
            "cmrr_db": 85 * noise_factor,
            "psrr_db": 75 * noise_factor,
            "input_noise_nv_sqrt_hz": 12 * noise_factor,
            "output_swing_v": c.supply_voltage * 0.7,
            "slew_rate_v_us": 50 * noise_factor
        }
        
        self.log(f"  DC Gain:       {results['dc_gain_db']:.1f} dB")
        self.log(f"  GBW:           {results['gbw_mhz']:.1f} MHz")
        self.log(f"  Phase Margin:  {results['phase_margin_deg']:.1f}°")
        self.log(f"  Power:         {results['power_mw']:.2f} mW")
        self.log(f"  CMRR:          {results['cmrr_db']:.1f} dB")
        
        passed = (
            results["dc_gain_db"] >= c.dc_gain_min and
            results["gbw_mhz"] >= c.gbw_min and
            results["phase_margin_deg"] >= c.phase_margin_min and
            results["power_mw"] <= c.power_max
        )
        
        self.log(f"  {'ALL SPECS MET' if passed else 'SPECS NOT MET'}")
        
        return {
            "status": "success" if passed else "failed",
            "metrics": results,
            "specs_met": passed,
            "violations": self._check_violations(results, c) if not passed else []
        }
    
    def _check_violations(self, results, constraints):
        violations = []
        if results["dc_gain_db"] < constraints.dc_gain_min:
            violations.append(f"DC gain: {results['dc_gain_db']:.1f} < {constraints.dc_gain_min} dB")
        if results["gbw_mhz"] < constraints.gbw_min:
            violations.append(f"GBW: {results['gbw_mhz']:.1f} < {constraints.gbw_min} MHz")
        if results["phase_margin_deg"] < constraints.phase_margin_min:
            violations.append(f"PM: {results['phase_margin_deg']:.1f} < {constraints.phase_margin_min}°")
        if results["power_mw"] > constraints.power_max:
            violations.append(f"Power: {results['power_mw']:.2f} > {constraints.power_max} mW")
        return violations


print("Agent classes defined: TopologyAgent, SizingAgent, SimulationAgent")
print("Each agent implements the BaseAgent interface with typed state management.")

In [ ]:
class SupervisorAgent(BaseAgent):
    """Central orchestrator that decomposes intent and manages the design loop."""
    
    MAX_ITERATIONS = 5
    
    def __init__(self):
        super().__init__(AgentRole.SUPERVISOR, model="gpt-5.2")
        self.workers = {
            AgentRole.TOPOLOGY: TopologyAgent(),
            AgentRole.SIZING: SizingAgent(),
            AgentRole.SIMULATION: SimulationAgent(),
        }
    
    def execute(self, task: AgentTask, state: DesignState) -> dict:
        self.log(f"Received intent: {state.intent}")
        self.log(f"Decomposing into sub-tasks...")
        
        for iteration in range(self.MAX_ITERATIONS):
            state.iteration_count = iteration + 1
            print(f"\n{'='*60}")
            print(f"  ITERATION {iteration + 1}/{self.MAX_ITERATIONS}")
            print(f"{'='*60}")
            
            # Step 1: Topology Selection
            if state.topology is None:
                topo_task = AgentTask(assigned_agent=AgentRole.TOPOLOGY,
                                     description="Select optimal topology")
                topo_result = self.workers[AgentRole.TOPOLOGY].execute(topo_task, state)
                if topo_result["status"] == "success":
                    state.topology = topo_result["topology"]
                else:
                    return {"status": "failed", "reason": "No viable topology"}
            
            # Step 2: Transistor Sizing
            sizing_task = AgentTask(assigned_agent=AgentRole.SIZING,
                                   description="Compute W/L ratios")
            sizing_result = self.workers[AgentRole.SIZING].execute(sizing_task, state)
            
            # Step 3: Simulation & Verification
            sim_task = AgentTask(assigned_agent=AgentRole.SIMULATION,
                                description="Run SPICE simulation")
            sim_result = self.workers[AgentRole.SIMULATION].execute(sim_task, state)
            state.simulation_results = sim_result.get("metrics")
            
            # Step 4: Decision
            if sim_result.get("specs_met", False):
                state.converged = True
                self.log(f"\n  DESIGN CONVERGED in {iteration + 1} iteration(s)!")
                state.ppa_metrics = sim_result["metrics"]
                return {"status": "converged", "iterations": iteration + 1, 
                        "metrics": sim_result["metrics"]}
            else:
                violations = sim_result.get("violations", [])
                self.log(f"  Violations: {violations}")
                self.log(f"  → Iterating with adjusted parameters...")
                state.history.append({
                    "iteration": iteration + 1,
                    "metrics": sim_result["metrics"],
                    "violations": violations
                })
        
        return {"status": "max_iterations", "final_metrics": state.simulation_results}


# Run the Supervisor-Worker system
print("\n" + "#"*60)
print("  SUPERVISOR-WORKER MAS EXECUTION")
print("#"*60)

supervisor = SupervisorAgent()
random.seed(42)

main_task = AgentTask(
    assigned_agent=AgentRole.SUPERVISOR,
    description="Design complete OTA",
    constraints=constraints
)

result = supervisor.execute(main_task, state)

print(f"\n{'='*60}")
print(f"  FINAL RESULT: {result['status'].upper()}")
if 'metrics' in result:
    print(f"  Iterations: {result.get('iterations', 'N/A')}")
    for k, v in result['metrics'].items():
        print(f"    {k}: {v}")
print(f"{'='*60}")

## 2.3 Consensus-Based Reasoning — Multi-Topology Exploration

### The Problem with Single-Path Design

The Supervisor-Worker pattern selects one topology and iterates. But what if the **wrong topology** was chosen? In analog design, topology selection is often the most critical decision — a bad topology cannot be "fixed" by better sizing.

### Consensus Architecture

Consensus-Based Reasoning addresses this by:

1. **Spawning multiple divergent agents**, each exploring a different topology
2. **Simulating all candidates** in parallel
3. **Aggregating results** through a PPA Arbiter that performs multi-objective optimization

This is directly inspired by **EDAid's Divergent Thought Collaboration**, where different Chain-of-Thought (CoT) prompts prevent the system from getting stuck in suboptimal pathways.

### Formal Model

Given $k$ divergent agents with proposals $\{p_1, \ldots, p_k\}$ and PPA evaluation function $\Phi$:

$$p^* = \arg\min_{p_i} \sum_{j} w_j \cdot \left(\frac{\Phi_j(p_i) - \Phi_j^{\text{target}}}{\Phi_j^{\text{target}}}\right)^2$$

Where $w_j$ are importance weights for each PPA dimension.

In [ ]:
import numpy as np
import plotly.graph_objects as go


class DivergentTopologyAgent:
    """An agent that explores one specific topology with a unique CoT prompt."""
    
    def __init__(self, topology_name: str, cot_style: str):
        self.topology = topology_name
        self.cot_style = cot_style
    
    def propose(self, constraints: DesignConstraints) -> dict:
        """Generate a design proposal with simulated PPA metrics."""
        np.random.seed(hash(self.topology) % 2**32)
        
        topology_profiles = {
            "telescopic_cascode": {
                "gain": 72, "gbw": 250, "power": 0.3, "area": 0.008,
                "pm": 75, "cmrr": 90, "swing": 0.8
            },
            "folded_cascode": {
                "gain": 65, "gbw": 180, "power": 0.5, "area": 0.012,
                "pm": 70, "cmrr": 85, "swing": 1.2
            },
            "two_stage_miller": {
                "gain": 85, "gbw": 120, "power": 0.7, "area": 0.015,
                "pm": 62, "cmrr": 80, "swing": 1.4
            },
            "gain_boosted": {
                "gain": 110, "gbw": 90, "power": 1.2, "area": 0.025,
                "pm": 55, "cmrr": 95, "swing": 1.0
            }
        }
        
        base = topology_profiles.get(self.topology, topology_profiles["folded_cascode"])
        noise = {k: v * np.random.uniform(0.9, 1.1) for k, v in base.items()}
        
        return {
            "topology": self.topology,
            "cot_style": self.cot_style,
            "metrics": noise
        }


class PPAArbiter:
    """Multi-objective arbiter that selects the Pareto-optimal design."""
    
    def __init__(self, weights: dict):
        self.weights = weights
    
    def evaluate(self, proposals: list[dict], constraints: DesignConstraints) -> dict:
        scores = []
        
        for p in proposals:
            m = p["metrics"]
            score = (
                self.weights.get("gain", 0.2) * min(m["gain"] / constraints.dc_gain_min, 1.5) +
                self.weights.get("gbw", 0.2) * min(m["gbw"] / constraints.gbw_min, 1.5) +
                self.weights.get("power", 0.2) * (1 - m["power"] / constraints.power_max) +
                self.weights.get("pm", 0.15) * min(m["pm"] / constraints.phase_margin_min, 1.5) +
                self.weights.get("area", 0.1) * (1 - m["area"] / 0.03) +
                self.weights.get("cmrr", 0.1) * min(m["cmrr"] / 80, 1.5) +
                self.weights.get("swing", 0.05) * min(m["swing"] / 1.0, 1.5)
            )
            scores.append(score)
        
        return {
            "selected_idx": int(np.argmax(scores)),
            "scores": scores,
            "ranking": [proposals[i]["topology"] for i in np.argsort(scores)[::-1]]
        }


# Run consensus-based topology exploration
divergent_agents = [
    DivergentTopologyAgent("telescopic_cascode", "power-first reasoning"),
    DivergentTopologyAgent("folded_cascode", "balanced tradeoff reasoning"),
    DivergentTopologyAgent("two_stage_miller", "gain-first reasoning"),
    DivergentTopologyAgent("gain_boosted", "maximum-gain reasoning"),
]

proposals = [agent.propose(constraints) for agent in divergent_agents]

arbiter = PPAArbiter(weights={
    "gain": 0.25, "gbw": 0.20, "power": 0.20, 
    "pm": 0.15, "area": 0.10, "cmrr": 0.05, "swing": 0.05
})

decision = arbiter.evaluate(proposals, constraints)

print("\n" + "="*60)
print("  CONSENSUS-BASED TOPOLOGY SELECTION")
print("="*60)

for i, (p, s) in enumerate(zip(proposals, decision["scores"])):
    marker = "→" if i == decision["selected_idx"] else " "
    print(f"  {marker} [{s:.3f}] {p['topology']:25s} (CoT: {p['cot_style']})")
    m = p["metrics"]
    print(f"           Gain={m['gain']:.0f}dB  GBW={m['gbw']:.0f}MHz  Power={m['power']:.2f}mW  PM={m['pm']:.0f}°")

print(f"\n  Winner: {proposals[decision['selected_idx']]['topology']}")
print(f"  Ranking: {' > '.join(decision['ranking'])}")

In [ ]:
categories = ['DC Gain', 'GBW', 'Power Eff.', 'Phase Margin',
               'CMRR', 'Output Swing', 'Area Eff.']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

radar_colors = [TEAL, BLUE, RED, YELLOW]

for i, p in enumerate(proposals):
    m = p['metrics']
    values = [
        m['gain'] / 120, m['gbw'] / 300, 1 - m['power'] / 1.5,
        m['pm'] / 90, m['cmrr'] / 100, m['swing'] / 1.5,
        1 - m['area'] / 0.03
    ]
    values = [max(0, min(1, v)) for v in values]
    values += values[:1]

    is_winner = (i == decision['selected_idx'])
    lw = 3.0 if is_winner else 1.5
    alpha_fill = 0.18 if is_winner else 0.05

    if is_winner:
        ax.plot(angles, values, 'o-', linewidth=lw + 4, alpha=0.10,
                color=radar_colors[i], markersize=0)
        ax.plot(angles, values, 'o-', linewidth=lw + 2, alpha=0.18,
                color=radar_colors[i], markersize=0)
    ax.plot(angles, values, 'o-', linewidth=lw, color=radar_colors[i],
            label=f"{p['topology']} ({decision['scores'][i]:.3f})",
            markersize=5)
    ax.fill(angles, values, alpha=alpha_fill, color=radar_colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10, color=TEXT)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8, color=TEXT_DIM)
ax.grid(color=GRID, alpha=0.5)
ax.set_facecolor(BACKGROUND)

ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10,
          facecolor=SURFACE, edgecolor=GRID, labelcolor=TEXT,
          title='Topology (Score)', title_fontsize=11)

ax.set_title('Consensus-Based Topology Comparison\n(Normalized PPA Metrics)',
             fontsize=14, fontweight='bold', color=TEXT, pad=30)

finish_plot(fig, ax)
plt.show()

## 2.4 The Handoff Pattern — Dynamic Control Flow

### Architecture

The Handoff Pattern passes **control and context** dynamically between specialized agents. Unlike Supervisor-Worker (where the supervisor always holds control), handoff allows any agent to transfer to any other agent based on runtime conditions.

### Key Properties

1. **Error Context Propagation**: When a Sizing Agent fails to meet gain requirements, it hands the state back to the Topology Selection Agent with **specific error context** explaining why the current topology is insufficient
2. **Dynamic Routing**: The next agent is determined at runtime, not at compile time
3. **State Continuity**: The full design state transfers with each handoff

### Formal Model

A handoff is a function:

$$\text{Handoff}: (a_i, \mathcal{S}, \mathcal{E}_{\text{ctx}}) \rightarrow (a_j, \mathcal{S}')$$

Where $a_i$ is the current agent, $\mathcal{E}_{\text{ctx}}$ is the error/success context, and $a_j$ is the receiving agent with updated state $\mathcal{S}'$.

In [ ]:
from dataclasses import dataclass
from typing import Optional, Callable


@dataclass
class HandoffContext:
    """Context passed during a handoff between agents."""
    source_agent: str
    target_agent: str
    reason: str
    error_details: Optional[str] = None
    suggestions: list[str] = None
    state_snapshot: dict = None
    
    def __post_init__(self):
        if self.suggestions is None:
            self.suggestions = []
        if self.state_snapshot is None:
            self.state_snapshot = {}


class HandoffAgent:
    """Agent that supports dynamic handoff protocol."""
    
    def __init__(self, name: str, handler: Callable):
        self.name = name
        self.handler = handler
        self.handoff_history = []
    
    def process(self, state: dict, context: Optional[HandoffContext] = None) -> tuple:
        """Process state. Returns (updated_state, next_handoff_or_None)."""
        if context:
            print(f"  [{self.name}] Received handoff from {context.source_agent}")
            print(f"             Reason: {context.reason}")
            if context.suggestions:
                print(f"             Suggestions: {context.suggestions}")
        
        result = self.handler(state, context)
        self.handoff_history.append(result)
        return result


def topology_handler(state, context):
    if context and "increase gain" in (context.reason or "").lower():
        print(f"  [Topology] Switching from {state.get('topology', 'none')} to gain_boosted_cascode")
        state["topology"] = "gain_boosted_cascode"
        state["topology_changes"] = state.get("topology_changes", 0) + 1
    else:
        state["topology"] = "two_stage_miller"
    
    return state, HandoffContext(
        source_agent="Topology", target_agent="Sizing",
        reason="Topology selected, proceed to sizing",
        state_snapshot={"topology": state["topology"]}
    )


def sizing_handler(state, context):
    topology = state.get("topology", "unknown")
    print(f"  [Sizing] Computing sizes for {topology}")
    
    gain_achieved = {"two_stage_miller": 55, "gain_boosted_cascode": 75}.get(topology, 50)
    state["gain_achieved"] = gain_achieved
    state["sizing_done"] = True
    
    if gain_achieved < 60:
        print(f"  [Sizing] Gain {gain_achieved}dB < 60dB target — handing back to Topology")
        return state, HandoffContext(
            source_agent="Sizing", target_agent="Topology",
            reason="Increase gain — current topology insufficient",
            error_details=f"Achieved {gain_achieved}dB, need ≥60dB",
            suggestions=["Consider gain-boosted cascode", "Add regulated cascode"]
        )
    
    print(f"  [Sizing] Gain {gain_achieved}dB meets target")
    return state, HandoffContext(
        source_agent="Sizing", target_agent="Simulation",
        reason="Sizing complete, proceed to simulation"
    )


def simulation_handler(state, context):
    print(f"  [Simulation] Running SPICE with gain={state.get('gain_achieved', 0)}dB")
    state["simulated"] = True
    state["final_gain"] = state.get("gain_achieved", 0) * 1.05
    return state, None  # Terminal — no further handoff


# Build the handoff chain
agents_map = {
    "Topology": HandoffAgent("Topology", topology_handler),
    "Sizing": HandoffAgent("Sizing", sizing_handler),
    "Simulation": HandoffAgent("Simulation", simulation_handler),
}

# Execute the handoff chain
print("\n" + "#"*60)
print("  HANDOFF PATTERN EXECUTION")
print("#"*60)

state = {}
current_agent = "Topology"
context = None
step = 0

while current_agent and step < 10:
    step += 1
    print(f"\n--- Step {step}: {current_agent} ---")
    
    agent = agents_map[current_agent]
    state, next_context = agent.process(state, context)
    
    if next_context:
        current_agent = next_context.target_agent
        context = next_context
    else:
        print(f"\n  Chain complete. Final state:")
        for k, v in state.items():
            print(f"     {k}: {v}")
        break

## 2.5 Stateful Graph Workflows -- DAGs and Cyclic Loops

### The Need for Graph-Based Design Flows

Real EDA flows are neither purely linear nor purely tree-structured.
They contain:

- **Parallel branches** (synthesis and verification can run concurrently)
- **Conditional routing** (if timing fails, re-synthesize; if DRC fails, re-place)
- **Feedback loops** (iterative refinement until convergence)
- **Checkpoints** (save state before expensive operations)

These are naturally modeled as **Directed Graphs** -- state machines where:
- **Nodes** represent agent actions or decision points
- **Edges** represent transitions with conditions
- **State** is a typed dictionary that evolves through the graph

### Graph Topology Classification

| Type | Properties | EDA Use Case |
|------|-----------|-------------|
| **DAG** | Acyclic, parallel branches | Initial design decomposition |
| **Cyclic** | Contains feedback loops | Iterative optimization |
| **Hierarchical** | Nested subgraphs | Block-level to chip-level design |

### Example EDA Workflow Graph

**Nodes and edges of the stateful graph built in the next cell:**

- **Intent** feeds into the **Supervisor** (entry node)
- **Supervisor** routes to **Synthesis** (default edge)
- **Synthesis** routes to **Verification** (default edge)
- **Verification** is a conditional branch:
  - On **pass**: route to **Layout** (physical design)
  - On **fail**: loop back to **Supervisor** (re-iterate)
- **Layout** routes to **Closure** (design sign-off)
- **Closure** routes to **end** (terminal node)

In [ ]:
from typing import TypedDict, Literal
from dataclasses import dataclass, field
import random


class GraphState(TypedDict):
    """Typed state for the EDA workflow graph."""
    intent: str
    topology: str
    netlist: str
    sim_results: dict
    drc_clean: bool
    lvs_clean: bool
    iteration: int
    decision: str
    log: list[str]
    trace: list[tuple]


class EDAWorkflowGraph:
    """A minimal graph-based EDA workflow engine.

    Demonstrates the core concepts that LangGraph provides:
    - Nodes as agent functions
    - Edges as conditional transitions
    - State persistence across transitions
    - Execution trace recording for post-hoc analysis
    """

    def __init__(self):
        self.nodes: dict[str, callable] = {}
        self.edges: dict[str, list[tuple[callable, str]]] = {}
        self.default_edges: dict[str, str] = {}
        self.entry_node: str = ""

    def add_node(self, name: str, func: callable):
        self.nodes[name] = func

    def add_edge(self, source: str, target: str):
        self.default_edges[source] = target

    def add_conditional_edge(self, source: str, condition: callable,
                             targets: dict):
        if source not in self.edges:
            self.edges[source] = []
        self.edges[source].append((condition, targets))

    def set_entry(self, name: str):
        self.entry_node = name

    def run(self, initial_state: dict, max_steps: int = 20) -> dict:
        state = dict(initial_state)
        current = self.entry_node
        step = 0

        while current and current != "__end__" and step < max_steps:
            step += 1
            print(f"\n  Step {step}: [{current}]")

            node_func = self.nodes[current]
            state = node_func(state)

            next_node = None
            if current in self.edges:
                for condition, targets in self.edges[current]:
                    result = condition(state)
                    if result in targets:
                        next_node = targets[result]
                        print(f"           -> Conditional: "
                              f"{result} -> {next_node}")
                        break

            if next_node is None and current in self.default_edges:
                next_node = self.default_edges[current]
                print(f"           -> Default -> {next_node}")

            if next_node == "__end__" or next_node is None:
                outcome = "terminal"
            else:
                outcome = state.get("decision", "ok")
            state.setdefault("trace", []).append((current, outcome))

            current = next_node

        return state


def supervisor_node(state):
    state["iteration"] = state.get("iteration", 0) + 1
    state["log"] = state.get("log", [])
    state["log"].append(
        f"Iteration {state['iteration']}: Supervisor decomposing task")
    print(f"           Supervisor: iteration {state['iteration']}")
    return state


def synthesis_node(state):
    state["netlist"] = f"netlist_v{state['iteration']}"
    state["log"].append(f"Synthesis: generated {state['netlist']}")
    print(f"           Synthesis: generated {state['netlist']}")
    return state


def verification_node(state):
    passed = random.random() > 0.4 or state["iteration"] >= 3
    state["sim_results"] = {
        "passed": passed,
        "gain": 60 + state["iteration"] * 5
    }
    state["decision"] = "pass" if passed else "fail"
    status_str = "PASS" if passed else "FAIL"
    gain_val = state["sim_results"]["gain"]
    state["log"].append(
        f"Verification: {status_str} (gain={gain_val}dB)")
    print(f"           Verification: {status_str} "
          f"(gain={gain_val}dB)")
    return state


def layout_node(state):
    state["drc_clean"] = True
    state["lvs_clean"] = True
    state["log"].append("Layout: DRC/LVS clean")
    print(f"           Layout: DRC clean={state['drc_clean']}, "
          f"LVS clean={state['lvs_clean']}")
    return state


def closure_node(state):
    state["log"].append(
        f"Design closure achieved in {state['iteration']} iterations")
    print(f"           DESIGN CLOSURE in {state['iteration']} "
          f"iterations!")
    return state


# Build the graph
graph = EDAWorkflowGraph()
graph.add_node("supervisor", supervisor_node)
graph.add_node("synthesis", synthesis_node)
graph.add_node("verification", verification_node)
graph.add_node("layout", layout_node)
graph.add_node("closure", closure_node)

graph.set_entry("supervisor")
graph.add_edge("supervisor", "synthesis")
graph.add_edge("synthesis", "verification")
graph.add_conditional_edge(
    "verification",
    lambda s: s["decision"],
    {"pass": "layout", "fail": "supervisor"}
)
graph.add_edge("layout", "closure")
graph.add_edge("closure", "__end__")

# Run
print("\n" + "#" * 60)
print("  STATEFUL GRAPH WORKFLOW EXECUTION")
print("#" * 60)

random.seed(42)
initial = {"intent": "Design OTA", "iteration": 0}
final_state = graph.run(initial)

print(f"\n{'=' * 60}")
print("  Execution Log:")
for entry in final_state.get("log", []):
    print(f"    - {entry}")
print(f"{'=' * 60}")

## 2.6 Pattern Selection Decision Framework

### Choosing the Right Pattern

The selection of an MAS pattern depends on the specific EDA task characteristics:

| Design Task | Recommended Pattern | Rationale |
|-------------|-------------------|----------|
| Full chip top-down design | **Supervisor-Worker** | Central coordination, parallel block design |
| Topology exploration | **Consensus-Based** | Multiple valid alternatives, need Pareto selection |
| Sequential pipeline (spec→schematic→layout) | **Handoff** | Each stage depends on prior stage output |
| Iterative optimization with constraints | **Stateful Graph** | Conditional loops, checkpointing |
| Mixed: exploration + refinement | **Hybrid** | Consensus for exploration, graph for refinement |

### Hybrid Architectures

Production systems typically combine patterns:

1. **Outer loop**: Stateful Graph managing the overall design flow
2. **Topology node**: Uses Consensus-Based reasoning internally
3. **Sizing node**: Uses Supervisor-Worker with multiple sizing agents
4. **Verification node**: Uses Handoff to escalate failures

This compositional approach gives maximum flexibility while maintaining clear boundaries.

In [ ]:
patterns = ['Supervisor-\nWorker', 'Consensus-\nBased', 'Handoff',
            'Stateful\nGraph', 'Hybrid']
metrics = {
    'Latency': [3, 5, 2, 4, 4],
    'Fault Tolerance': [4, 5, 2, 4, 5],
    'Design Quality': [3, 5, 3, 4, 5],
    'Complexity': [3, 4, 2, 5, 5],
    'Parallelism': [4, 5, 1, 3, 5],
}

fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(patterns))
width = 0.15
bar_colors = [TEAL, BLUE, RED, YELLOW, PURPLE]

for i, (metric, values) in enumerate(metrics.items()):
    offset = x + i * width - 2 * width
    ax.bar(offset, values, width, color=bar_colors[i], alpha=0.12,
           linewidth=0)
    ax.bar(offset, values, width, color=bar_colors[i], alpha=0.25,
           linewidth=0)
    ax.bar(offset, values, width, label=metric,
           color=bar_colors[i], alpha=0.85, edgecolor=bar_colors[i],
           linewidth=0.8)

ax.set_xlabel('Architecture Pattern', fontsize=12, fontweight='bold')
ax.set_ylabel('Score (1-5)', fontsize=12, fontweight='bold')
ax.set_title('MAS Pattern Comparison Matrix',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(patterns, fontsize=10)
ax.legend(fontsize=9, loc='upper left')
ax.set_ylim(0, 6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

finish_plot(fig, ax)
plt.show()

## 2.7 Summary and Key Takeaways

### Pattern Mastery Checklist

- [x] **Supervisor-Worker**: Central orchestration with task decomposition and parallel worker execution
- [x] **Consensus-Based**: Multi-agent topology exploration with PPA arbitration
- [x] **Handoff**: Dynamic control flow with error context propagation between specialists
- [x] **Stateful Graph**: DAG/cyclic workflows with conditional routing and checkpointing

### Design Principles

1. **Type your state** — Use typed dictionaries/dataclasses for all shared state
2. **Separate generation from critique** — Generator agents should never self-evaluate
3. **Propagate error context** — Failed handoffs must include actionable error details
4. **Design for iteration** — Convergence is the goal, not one-shot correctness
5. **Compose patterns** — Use hybrid architectures for real systems

### What's Next

In **Chapter 3**, we implement these patterns using **LangGraph** — the production framework for stateful graph workflows — including persistent checkpointing, time-travel debugging, and conditional routing for long-running EDA simulations.

---

*"The art of agent orchestration is not in building smarter individual agents, but in designing the communication protocol between them."*

In [ ]:
# Trace Analysis: Visualise the execution trace of the DAG

trace = final_state.get("trace", [])

print("Execution Trace:")
print("-" * 50)
for step_idx, (node_name, outcome) in enumerate(trace):
    marker = "->" if outcome != "terminal" else "[]"
    print(f"  Step {step_idx}: {node_name:15s}  {marker}  "
          f"outcome = '{outcome}'")

print(f"\nTotal steps executed: {len(trace)}")
fail_count = sum(1 for _, o in trace if o == "fail")
print(f"Back-edge traversals (fail loops): {fail_count}")

# Timeline visualization with 3B1B glow styling
fig, ax = plt.subplots(figsize=(14, 3))

node_colors = {
    "supervisor": BLUE,
    "synthesis": TEAL,
    "verification": YELLOW,
    "layout": PURPLE,
    "closure": GREEN,
}

for i, (node_name, outcome) in enumerate(trace):
    color = node_colors.get(node_name, TEXT_DIM)
    ax.barh(0, 0.9, left=i + 0.05, color=color, alpha=0.10,
            height=0.85, linewidth=0)
    ax.barh(0, 0.9, left=i + 0.05, color=color, alpha=0.22,
            height=0.6, linewidth=0)
    ax.barh(0, 0.9, left=i + 0.05, color=color, alpha=0.85,
            edgecolor=color, height=0.42, linewidth=1.5)
    ax.text(i + 0.5, 0, node_name, ha="center", va="center",
            fontsize=7, fontweight="bold", color=TEXT, rotation=45)
    if outcome == "fail":
        ax.text(i + 0.5, -0.38, "FAIL", ha="center", va="center",
                fontsize=6, color=RED, fontweight="bold")

ax.set_xlim(-0.2, max(len(trace), 1) + 0.2)
ax.set_ylim(-0.6, 0.6)
ax.set_xlabel("Execution Step")
ax.set_title("DAG Execution Timeline",
             fontsize=13, fontweight="bold")
ax.set_yticks([])
ax.spines['left'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

finish_plot(fig, ax)
plt.show()